In [1]:
from langchain_openai import ChatOpenAI
import dotenv
from pathlib import Path

class Config:
    models_with_structured_output_support = {'azure-gpt-4o', 'claude-3-5-sonnet', 'gemini-1.5-pro'}
    env_config = dotenv.dotenv_values(Path(".env"))
    dotenv.load_dotenv(Path(".env"))

llm = ChatOpenAI(
    model="azure-o3-mini",
    base_url=Config.env_config['OPENAI_BASE_URL'],
    api_key=Config.env_config['OPENAI_API_KEY'],
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    seed=1000
)

In [2]:
from typing_extensions import TypedDict
from operator import add
from typing import Annotated, List

class State(TypedDict):
    current_section: str
    steps: Annotated[List[str], add]
    response: str

In [30]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

class GenerateReport():

    generate_report_system_prompt = '''
    <Instructions>
    The user will input an outline for a manuscript on a specific topic. You are a scholarly ghost‑writer with a PhD in that topic area. Your task is to convert a detailed, hierarchically coded outline into polished manuscript prose. Follow the following global constraints for every section you draft.
 
    **Voice & register**
    
    - Doctoral‑level, formal scientific style (as in a peer‑reviewed journal).
    - Integrate definitions, mechanisms, empirical findings and theoretical nuance as appropriate.
    
    **Form**

    - Only paragraphs — no headings, no numbering, no bullet points, no embedded outline codes.
    - You may use multiple paragraphs if needed to cover the content deeply and coherently.
    
    **Use of outline**
    
    - Each outline line is structured as “<alphanumeric code> -> <level‑1 topic> -> <level‑2 topic> -> …”.
    - Begin every new answer by quoting, verbatim, the last ontological branch term (i.e, only the text after the last sequential “ -> ”) that you are expanding. For example,  if the outline section heading you are working on is “<alphanumeric code> -> <level‑1 topic> -> <level‑2 topic> you would only quote <level‑2 topic> with its alphanumeric designation
    - Treat higher‑level nodes as contextual background, lower‑level nodes as focal content.
    - Avoid repeating detailed explanations already supplied for earlier sections unless essential for clarity; instead, use concise forward/backward references if necessary.
    - Confidence scoring: For each factual statement generated, include a confidence score between 1-10 (1 being the lowest and 10 being the highest) in the form of “(CS= [score])”. This score should reflect the level of scientific consensus or evidence supporting the statement, with 1 indicating speculative or weakly supported claims and 10 indicating well-established facts.
    </Instructions>
    '''

    generate_report_human_prompt = '''
    <Current Section>
    {current_section}
    </Current Section>
    
    Write under the provided subsection. Output in the following format.
    ```json
    {{
        "<last ontological branch term>": "<content>"    
    }}
    ```
    '''

    # -----------------------------------------------------------------------
    generate_report_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                (
                    generate_report_system_prompt
                ),
            ),
            (
                "human",
                (
                    generate_report_human_prompt
                ),
            ),
        ]
    )

    def __init__(self, llm):
        self.generate_report_chain = self.generate_report_prompt | llm | JsonOutputParser()

    def __call__(self, state: State):
        '''LLM generates reports from a given outline'''
        response = self.generate_report_chain.invoke(input={'current_section': state['current_section']})
        return {'response': response, 'steps': ['Generate Report']}

In [46]:
class AddCitations():

    add_citations_system_prompt = '''
    <Instructions>
    You are given a paragraph that represents discussion on a certain topic. You are an expert on that topic. You will tag the particular portion of the text that needs reference.

    1. You will find the line(s) that needs reference. A line needs reference if it contains information that was result of previous studies.
    2. You will wrap those line(s) with curly braces "CITE()". Example: CITE(This line needs reference).
    4. Do not add anything else to the input except the "CITE()".
    3. You will output the original input with the added curly braces.

    </Instructions>
    '''

    add_citations_human_prompt = '''
    <Input Paragraph>
    {input_paragraph}
    </Input Paragraph>
    
    Tag the text from the above input paragraph that need references.
    '''

    # -----------------------------------------------------------------------
    add_citations_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                (
                    add_citations_system_prompt
                ),
            ),
            (
                "human",
                (
                    add_citations_human_prompt
                ),
            ),
        ]
    )

    def __init__(self, llm):
        self.add_citations_chain = self.add_citations_prompt | llm

    def __call__(self, state: State):
        '''LLM generates reports from a given outline'''
        response = self.add_citations_chain.invoke(input={'input_paragraph': list(state['response'].values())[0]}).content
        return {'response': response, 'steps': ['Add Citations']}

In [47]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, END, StateGraph

# Define a new graph
workflow = StateGraph(state_schema=State)

# Define the (single) node in the graph
workflow.add_node("Generate Report", GenerateReport(llm))
workflow.add_node("Add Citations", AddCitations(llm))
workflow.add_edge(START, "Generate Report")
workflow.add_edge("Generate Report", "Add Citations")
workflow.add_edge("Add Citations", END)

# Add memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [ ]:
from utils import agent
res = agent.invoke({'current_section': 'Title: Hypertensive Disorders of Pregnancy: A Comprehensive Review of Pathophysiology, Clinical Management, Long-Term Implications, and Future Directions -> I. Introduction'},
                      {"configurable": {"thread_id": "abc124"}})['response']
print(res)

Hypertensive disorders of pregnancy represent a spectrum of conditions, including gestational hypertension, preeclampsia, and eclampsia, all of which have been identified as significant contributors to maternal and perinatal morbidity and mortality (CS=10).

This introduction provides a foundational context by delineating the clinical significance and epidemiological impact of these disorders.

CITE(By definition, hypertensive disorders of pregnancy involve elevated blood pressure arising during gestation, often accompanied by multi‐system complications that pose complex challenges for clinicians and researchers alike (CS=10).)

CITE(Moreover, the pathophysiological mechanisms underpinning these conditions—ranging from abnormal placentation to systemic endothelial dysfunction—highlight the intricate interplay between maternal and fetal health (CS=9).)

CITE(The present review aims to integrate current empirical findings and clinical insights to offer a comprehensive evaluation of the d

In [116]:
from collections import OrderedDict
def dfs(report, secs, content):
    if len(secs) == 0: return OrderedDict({'content': content})
    report[secs[0]] = dfs(report[secs[0]] if secs[0] in report else OrderedDict(), secs[1:], content)
    return report

with open('data/outline.txt') as f:
    sts = [st.strip() for st in f.readlines() if st.strip() != '']

report = OrderedDict()
for st in sts:
    response = app.invoke({'current_section': st}, {"configurable": {"thread_id": "abc123"}})['response']
    assert len(response) == 1, f"Invalid format: more than one subsection in the response, {response}"
    report = dfs(report, st.split(' -> '), list(response.values())[0])
    
print(report)

KeyboardInterrupt: 

In [117]:
pprint.pprint(report)


OrderedDict([('Title: Hypertensive Disorders of Pregnancy: A Comprehensive '
              'Review of Pathophysiology, Clinical Management, Long-Term '
              'Implications, and Future Directions',
              OrderedDict([('I. Introduction',
                            OrderedDict([('content',
                                          'Hypertensive disorders of pregnancy '
                                          'encompass a spectrum of clinical '
                                          'conditions characterized by '
                                          'elevated maternal blood pressure '
                                          'and systemic involvement that '
                                          'significantly impact both maternal '
                                          'and fetal outcomes (CS=10). In this '
                                          'context, the introduction '
                                          'establishes the clinical and public

In [111]:
import pprint
for report in report_list:
    pprint.pprint(report)

OrderedDict([('Title: Hypertensive Disorders of Pregnancy: A Comprehensive '
              'Review of Pathophysiology, Clinical Management, Long-Term '
              'Implications, and Future Directions',
              OrderedDict([('I. Introduction',
                            OrderedDict([('content',
                                          'Hypertensive disorders of pregnancy '
                                          '(HDP) represent a spectrum of '
                                          'conditions that complicate '
                                          'gestation and are associated with '
                                          'significant maternal and fetal '
                                          'morbidity and mortality (CS=10). '
                                          'These conditions, which include '
                                          'preeclampsia, gestational '
                                          'hypertension, and eclampsia, are '
        

In [85]:
from collections import OrderedDict

d = OrderedDict()
d[5] = 50
print(d | OrderedDict({2:20, 1:10}))

OrderedDict({5: 50, 2: 20, 1: 10})


In [84]:
for o in output:
    print(o)

Title: Hypertensive Disorders of Pregnancy: A Comprehensive Review of Pathophysiology, Clinical Management, Long-Term Implications, and Future Directions -> I. Introduction -> Hypertensive disorders of pregnancy (HDP) represent a multifaceted group of conditions that significantly contribute to maternal and perinatal morbidity and mortality (CS=10). The term encompasses a range of pathologies, including gestational hypertension, preeclampsia, and eclampsia, each characterized by distinct yet overlapping pathophysiological processes (CS=10). This review integrates current understanding from molecular mechanisms to clinical management, contextualizing the pathogenesis of HDP within the framework of abnormal placentation, angiogenic imbalance, and systemic endothelial dysfunction (CS=9).

Moreover, recognition of the chronic implications that these hypertensive disorders impart on long-term cardiovascular and renal health has spurred intensive research and clinical evaluation (CS=9). Cons

In [13]:
def dfs(p):
    if not p.next: return ""
    return {s: dfs(p.next[s]) for s in p.next}

class TrieNode:
    def __init__(self):
        self.next = {}

with open('data/test.txt') as f:
    sts = [st.strip() for st in f.readlines() if st.strip() != '']

root = TrieNode()
for st in sts:
    curr = root
    for s in st.split('; '):
        if s not in curr.next: curr.next[s] = TrieNode()
        curr = curr.next[s]

import json
with open('data/outline.json', 'w') as f:
    json.dump(dfs(root), f)